# Brent · `decision` (web: "Decision")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · All `decision` values

A `terms` aggregation on `decision.raw` (exact values, case-sensitive) plus a `missing`
aggregation for records with no decision. `size: 0` = no records, only the counts.
The `shown_as` column uses `repr()` so hidden characters (tabs, spaces) are visible.

In [2]:
body = {
    "size": 0,
    "track_total_hits": True,
    "query": brent_filter(),
    "aggs": {
        "decision": {"terms": {"field": "decision.raw", "size": 100}},
        "no_decision": {"missing": {"field": "decision.raw"}},
    },
}
data = search(body)

total = data["hits"]["total"]["value"]
buckets = data["aggregations"]["decision"]["buckets"]
decision_counts = pd.DataFrame(
    [(b["key"], b["doc_count"]) for b in buckets] + [(None, data["aggregations"]["no_decision"]["doc_count"])],
    columns=["decision", "count"],
).sort_values("count", ascending=False, ignore_index=True)

print("Brent applications:", total, "| sum of counts:", decision_counts["count"].sum())

Brent applications: 36952 | sum of counts: 36952


In [3]:
# From planninglondondatahub_public_technical_schemav2.1.xlsx, sheet "Field Values",
# index "applications", field "Decision" (extracted 2026-11-09).
SCHEMA_DECISION = [
    "Approved", "Called in by Secretary of State", "Closed", "Comment Issued",
    "Declined to Determine", "Not Required", "Refused", "Split Decision", "Withdrawn",
    "No Objection to Proposal (OBS only)", "Objection Raised to Proposal (OBS only)",
    "Opinion Issued", "Pre-application Advice Case Completed", "Referred to Mayor",
]

decision_counts["shown_as"] = decision_counts["decision"].map(repr)
decision_counts["in_schema"] = decision_counts["decision"].isin(SCHEMA_DECISION)
decision_counts["share_%"] = (100 * decision_counts["count"] / total).round(2)
decision_counts

,decision,count,shown_as,in_schema,share_%
0,Approved,26253,'Approved',True,71.05
1,Refused,8231,'Refused',True,22.27
2,Withdrawn,1417,'Withdrawn',True,3.83
3,NaN,1035,nan,False,2.80
4,Closed,6,'Closed',True,0.02
5,No Objection to Proposal (OBS only),4,'No Objection to Proposal (OBS only)',True,0.01
6,\tApproved,3,'\tApproved',False,0.01
7,Objection Raised to Proposal (OBS only),2,'Objection Raised to Proposal (OBS only)',True,0.01
8,Comment Issued,1,'Comment Issued',True,0.00


In [4]:
print("Seen in Brent but not in the schema:")
display(decision_counts[~decision_counts["in_schema"]])

unseen = sorted(set(SCHEMA_DECISION) - set(decision_counts["decision"].dropna()))
print(f"In the schema but never seen in Brent ({len(unseen)}):", unseen)

Seen in Brent but not in the schema:


,decision,count,shown_as,in_schema,share_%
3,NaN,1035,nan,False,2.80
6,\tApproved,3,'\tApproved',False,0.01


In the schema but never seen in Brent (7): ['Called in by Secretary of State', 'Declined to Determine', 'Not Required', 'Opinion Issued', 'Pre-application Advice Case Completed', 'Referred to Mayor', 'Split Decision']


### Findings · Step 1

Live API, 2026-11-09: **36,952** Brent applications, **8** distinct `decision` values
(plus empty). The counts add up to the total.

| Value | Count | Share | In schema? |
|---|---|---|---|
| `Approved` | 26,253 | 71.05% | yes |
| `Refused` | 8,231 | 22.27% | yes |
| `Withdrawn` | 1,417 | 3.83% | yes |
| empty (null) | 1,035 | 2.80% | no |
| `Closed` | 6 | 0.02% | yes |
| `No Objection to Proposal (OBS only)` | 4 | 0.01% | yes |
| `'\tApproved'` | 3 | 0.01% | **no** |
| `Objection Raised to Proposal (OBS only)` | 2 | 0.01% | yes |
| `Comment Issued` | 1 | 0.00% | yes |

- **Three values cover 97%:** Approved, Refused, Withdrawn.
- **Not in the schema:** empty (1,035) and `'\tApproved'` (3): `Approved` with a hidden
  tab in front, so an exact match on `Approved` misses these 3.
- **In the schema but never used in Brent (7):** `Called in by Secretary of State`,
  `Declined to Determine`, `Not Required`, `Opinion Issued`,
  `Pre-application Advice Case Completed`, `Referred to Mayor`, `Split Decision`.
- `Split Decision` is never a `decision` in Brent, but it is a `status` 22 times.
- **1,035 empty decisions vs 995 empty decision dates**: the two don't line up exactly
  (a check for Step 2).

## Step 2 · Applications with no decision: do they have a valid date and a decision date?

Fetch every Brent application with no `decision` (about a thousand, under the 10,000
window, so one request). For each: does it have a `valid_date`, a `decision_date`, and
what is its `status`? (An application still being considered has no decision yet, which
is normal.)

In [5]:
body = {
    "size": 5000,
    "track_total_hits": True,
    "query": {"bool": {
        "filter": [brent_filter()],
        "must_not": [{"exists": {"field": "decision.raw"}}],
    }},
    "_source": ["id", "lpa_app_no", "status", "valid_date", "decision_date"],
    "sort": [{"id": "asc"}],
}
data = search(body)
hits = data["hits"]["hits"]
assert data["hits"]["total"]["value"] == len(hits), "more records than one page"

no_decision = pd.DataFrame([h["_source"] for h in hits]).reindex(
    columns=["id", "lpa_app_no", "status", "valid_date", "decision_date"]
)
no_decision["has_valid_date"] = no_decision["valid_date"].notna()
no_decision["has_decision_date"] = no_decision["decision_date"].notna()
print("Applications with no decision:", len(no_decision))

pd.crosstab(no_decision["has_valid_date"], no_decision["has_decision_date"], margins=True)

Applications with no decision: 1035


has_decision_date,False,True,All
has_valid_date,,,
False,331,45,376
True,646,13,659
All,977,58,1035


In [6]:
# Their status, split by whether they have a decision date.
pd.crosstab(
    no_decision["status"].fillna("(empty)"), no_decision["has_decision_date"], margins=True
).sort_values("All", ascending=False)

has_decision_date,False,True,All
status,,,
All,977,58,1035
Application Received,459,0,459
(empty),407,11,418
Application Under Consideration,74,0,74
Superseded,1,27,28
Closed,21,0,21
Lapsed,0,15,15
Application Invalid,13,0,13
Allowed,2,1,3


### Findings · Step 2

**1,035** Brent applications have no `decision`.

| | no `decision_date` | has `decision_date` | total |
|---|---|---|---|
| no `valid_date` | 331 | 45 | 376 |
| has `valid_date` | 646 | 13 | 659 |
| **total** | **977** | **58** | **1,035** |

- **Mostly not a valid-date issue:** 659 of them have a valid date.
- **Most are missing the decision date too (977).** Only **58** have a decision date
  but no decision.

By `status`, they fall into three groups:

1. **Not decided yet, so no decision is normal (567):** `Application Received` 459,
   `Application Under Consideration` 74, `Closed` 21, `Application Invalid` 13. None of
   them has a decision date.
2. **Empty status (418):** the 407 shells (no decision date) and the 11 records with
   content (all 11 have a decision date). Already covered in `status`.
3. **Decided, but the decision is missing (50):** they have an outcome-type status and
   mostly a decision date: `Superseded` 28, `Lapsed`/`lapsed` 17, `Allowed` 3,
   `Dismissed` 2. The decision value itself was not recorded.

Hint for group 3 (not confirmed): the status implies the decision in most cases.
Only an approval can lapse, a superseded application was usually approved, and an
appeal (`Allowed` / `Dismissed`) follows a refusal.

## Step 3 · No decision date: how many also have no decision, and no status?

Counts with `_count`, each inside the Brent filter:
- no `decision_date`;
- no `decision_date` and no `decision`;
- no `decision_date`, no `decision` and no `status`;
- no `decision_date` but a `decision` (the other way round).

In [7]:
def count(must_not: list[str], must: list[str] = ()) -> int:
    query = {"bool": {
        "filter": [brent_filter(), *[{"exists": {"field": f}} for f in must]],
        "must_not": [{"exists": {"field": f}} for f in must_not],
    }}
    response = httpx.post(f"{BASE_URL}/applications/_count", headers=HEADERS, json={"query": query}, timeout=60)
    response.raise_for_status()
    return response.json()["count"]


pd.Series({
    "no decision_date": count(["decision_date"]),
    "no decision_date, no decision": count(["decision_date", "decision.raw"]),
    "no decision_date, no decision, no status": count(["decision_date", "decision.raw", "status.raw"]),
    "no decision_date, but has a decision": count(["decision_date"], must=["decision.raw"]),
}).to_frame("count")

,count
no decision_date,995
"no decision_date, no decision",977
"no decision_date, no decision, no status",407
"no decision_date, but has a decision",18


### Findings · Step 3

| | Count |
|---|---|
| no `decision_date` | 995 |
| … and no `decision` | **977** |
| … and no `decision`, and no `status` | **407** |
| no `decision_date`, but has a `decision` | 18 |

- **977 of the 995** applications without a decision date have no decision either. The
  other **18** have a decision but no date.
- **407 have no decision date, no decision and no status**: exactly the 407 shells from
  `status`. Nothing about the outcome is recorded for them.
- The rest of the 977 (570) have a status. From Step 2: 567 are not decided yet
  (Application Received, Under Consideration, Closed, Invalid), where no decision and no
  date is normal, and 3 are decided (`Superseded` 1, `Allowed` 2) with no decision and no
  date.

## Step 4 · The 18 with a decision but no decision date

Fetched with the columns the web app shows.

In [8]:
WEB_COLUMNS = [
    "lpa_app_no", "borough", "valid_date", "status", "decision", "decision_date",
    "application_type", "description", "site_name", "site_number", "street_name",
    "locality", "postcode", "decision_target_date", "appeal_decision",
    "appeal_decision_date", "url_planning_app",
]
body = {
    "size": 100,
    "track_total_hits": True,
    "query": {"bool": {
        "filter": [brent_filter(), {"exists": {"field": "decision.raw"}}],
        "must_not": [{"exists": {"field": "decision_date"}}],
    }},
    "_source": ["id", *WEB_COLUMNS],
    "sort": [{"id": "asc"}],
}
data = search(body)
hits = data["hits"]["hits"]
assert data["hits"]["total"]["value"] == len(hits), "more records than one page"

decision_no_date = pd.DataFrame([h["_source"] for h in hits]).reindex(columns=["id", *WEB_COLUMNS])
print("Decision but no decision date:", len(decision_no_date))
pd.set_option("display.max_colwidth", 60)
decision_no_date

Decision but no decision date: 18


,id,lpa_app_no,borough,valid_date,status,decision,decision_date,application_type,description,site_name,site_number,street_name,locality,postcode,decision_target_date,appeal_decision,appeal_decision_date,url_planning_app
0,Brent-04_1028,04/1028,Brent,NaN,Completed,Refused,None,All Other,Demolition of existing single-storey building on site an...,"LAND N/T 89, Harley Road, London, NW10",None,None,None,NaN,NaN,Allowed,04/11/2005,NaN
1,Brent-13_1179,13/1179,Brent,NaN,Completed,Refused,None,All Other,Demolition of 2 existing garages and erection of a singl...,"Yard rear of 46 Kempe Road and 41, Keslake Road, London,...",None,None,None,NW6 6SJ,NaN,Allowed,26/02/2014,NaN
2,Brent-16_5244,16/5244,Brent,NaN,Approved,Approved,None,All Other,Proposed part single storey and part double storey side ...,"St Nicholas Preparatory School, 22 Salmon Street, London...",None,None,None,NW9 8PN,NaN,NaN,NaN,NaN
3,Brent-18_4135,18/4135,Brent,31/10/2018,Withdrawn,Withdrawn,None,All Other,Change of use to a 4 bedroom HMO for up to 4 individuals...,"Flats 1-5 Inc, 7 Broadfield Close, London, NW2 6NR",None,None,None,NaN,19/12/2018,NaN,NaN,https://pa.brent.gov.uk/online-applications/applicationD...
4,Brent-19_1653,19/1653,Brent,16/05/2019,Withdrawn,Withdrawn,None,All Other,Retrospective planning application for the erection of a...,"21 Springfield Gardens, London, NW9 0RT",None,None,None,NaN,04/07/2019,NaN,NaN,https://pa.brent.gov.uk/online-applications/applicationD...
5,Brent-19_2377,19/2377,Brent,28/06/2019,Withdrawn,Withdrawn,None,Prior Approval,NaN,"46 Stafford Road, London, NW6 5YY",None,None,None,NaN,02/08/2019,NaN,NaN,https://pa.brent.gov.uk/online-applications/applicationD...
6,Brent-19_2970,19/2970,Brent,14/08/2019,Withdrawn,Withdrawn,None,Prior Approval,Prior approval for change of use from office (Use Class ...,"J M Amin & Co, 45 Ealing Road, Wembley, HA0 4BA",None,None,None,NaN,02/10/2019,NaN,NaN,https://pa.brent.gov.uk/online-applications/applicationD...
7,Brent-19_3296,19/3296,Brent,05/11/2019,Withdrawn,Withdrawn,None,All Other,Certificate of lawfulness for proposed construction of h...,"63 Oakington Avenue, Wembley, HA9 8HY",None,None,None,NaN,17/12/2019,NaN,NaN,https://pa.brent.gov.uk/online-applications/applicationD...
8,Brent-19_3998,19/3998,Brent,20/11/2019,Withdrawn,Withdrawn,None,Householder,Construction of a replacement single storey ground floor...,"34 Herbert Gardens, London, NW10 3BU",None,None,None,NaN,01/01/2020,NaN,NaN,https://pa.brent.gov.uk/online-applications/applicationD...
9,Brent-22_3218,22/3218,Brent,15/09/2022,Withdrawn,Withdrawn,None,All Other,Variation of Condition 2 (development carried out in acc...,"20A & 20B Farm Avenue, Wembley, HA0 4UU",None,None,None,NaN,03/11/2022,NaN,NaN,https://pa.brent.gov.uk/online-applications/applicationD...


### Findings · Step 4

The 18 fall into four groups:

**1. Withdrawn (10).** `decision` and `status` are both `Withdrawn`, valid dates
2018-2025, URL present, `CSV Data Import`. A withdrawal is the applicant stopping, not
a council decision, so no decision date is understandable.
(`Brent-18_4135`, `19_1653`, `19_2377`, `19_2970`, `19_3296`, `19_3998`, `22_3218`,
`22_3922`, `24_0736`, `25_0306`)

**2. Old applications from before the Datahub (4).** References from 1996-2016, no
`valid_date`, no URL, all last updated by `FME Excel Import`:

| id | status | decision | appeal |
|---|---|---|---|
| `Brent-96_1985` | `Appeal In Progress` | Refused | none recorded |
| `Brent-04_1028` | `Completed` | Refused | Allowed, 04/11/2005 |
| `Brent-13_1179` | `Completed` | Refused | Allowed, 26/02/2014 |
| `Brent-16_5244` | `Approved` | Approved | none |

`Brent-96_1985` (a 1996 application) still says `Appeal In Progress`.

**3. Contradictions (2).** `status` = `Application Under Consideration` but `decision`
= `Approved`: `Brent-25_3290` (outline application, 2-6 Hannah Close, valid
05/12/2025) and `Brent-26_1174` (variation of condition, former Gwalia Working Mens
Club, valid 22/07/2026). Either the decision was filled in early or the status is
behind.

**4. Test records (2).** `Brent-55_0003` ("*** TEST APPLICATION ***", Withdrawn) and
`Brent-55_0100` ("Test application record for system monitoring", Under Consideration,
decision Approved, appeal Allowed). Together with `Brent-55_0015` from `status`, all
three test records so far have a `55/` reference.

## Step 5 · Do the values match the schema type?

The schema gives `decision` the type `string`. Earlier counts used an Elasticsearch
aggregation (or `value_counts`), which can't show whether a value came as a JSON string or a
JSON number. Count the Python type of every value: `str` = JSON string, `int` / `float` =
JSON number, `NoneType` = null.

In [9]:
values, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "decision"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    values += [h["_source"] for h in page]
    after = page[-1]["sort"]

typed = pd.DataFrame(values).reindex(columns=["id", "decision"])
types = typed["decision"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
print("Brent applications:", len(typed))
display(types.value_counts().to_frame("count"))

not_str = typed[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["decision"].value_counts().head(30).to_frame("count")

Brent applications: 36952


,count
decision,
str,35917
NoneType,1035


values that are not strings: 0


,count
decision,


### Findings · Step 5

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 35,917 |
| null | 1,035 |
| number | 0 |

- **Every filled value is a JSON string**, as the schema says. No type mismatch.